In [ ]:
"""
Indicator Kriging Analysis of PRR Incidence
============================================
Adapts the geostatistical pipeline for Indicator Kriging (IK) using
binary presence/absence of disease (incidence) as the indicator variable.

Indicator variable: Z_i = 1 if plant i has severity >= INCIDENCE_THRESHOLD
                    Z_i = 0 otherwise

Pipeline per lot and evaluation period:
  1. Transform severity data to binary indicator (0/1)
  2. Compute empirical indicator variogram
  3. Fit theoretical models (Spherical, Exponential, Gaussian)
  4. Run Ordinary Kriging on indicator data (= Indicator Kriging)
  5. Post-process: clip estimated probabilities to [0, 1]
  6. Generate publication-quality figures

Lots
----
  - Donmatias  : Northing/Easting already in metres  (Taller_epi.xlsx)
  - El Retiro  : Latitude/Longitude                  (Data_ElRetiro_1338.xlsx)
  - La Ceja    : Latitude/Longitude                  (Data_LaCeja_1666.xlsx)

Outputs saved to OUTPUT_DIR
----------------------------
  Fig_IK_Variogram_<Lot>.png / .pdf  — indicator semivariograms (7 periods)
  Fig_IK_Maps_<Lot>.png / .pdf       — kriged probability maps  (7 periods)
  Fig_IK_Points_<Lot>.png / .pdf     — observed indicator maps  (7 periods)
  Table_IK_<Lot>.png / .pdf          — model comparison table
  indicator_variogram_params_<Lot>.csv

Dependencies
------------
    pip install pandas numpy scipy matplotlib openpyxl pykrige

Usage
-----
    python indicator_kriging_analysis.py
"""

import os
import warnings
import numpy  as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.optimize         import curve_fit
from scipy.spatial          import ConvexHull, Delaunay
from scipy.ndimage          import gaussian_filter
from pykrige.ok             import OrdinaryKriging
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
import matplotlib.patches   as mpatches
from matplotlib.colors import Normalize, LinearSegmentedColormap, BoundaryNorm
from matplotlib.cm     import ScalarMappable
from matplotlib.lines  import Line2D
warnings.filterwarnings("ignore")


# =============================================================================
# 0. USER CONFIGURATION
# =============================================================================

DONMATIAS_XLSX = "Taller_epi.xlsx"
EL_RETIRO_XLSX = "Data_ElRetiro_1338.xlsx"   # Data not included due to confidentiality
LA_CEJA_XLSX   = "Data_LaCeja_1666.xlsx"     # Data not included due to confidentiality

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Threshold: plants with severity >= this value are classified as PRESENT (1)
INCIDENCE_THRESHOLD = 1

# Evaluation periods (days)
PERIOD_BOUNDS = [
    ("P1",   0,  180),
    ("P2", 180,  360),
    ("P3", 360,  540),
    ("P4", 540,  720),
    ("P5", 720,  900),
    ("P6", 900, 1080),
    ("P7",1080, 1200),
]

# Variogram parameters
N_LAGS             = 12     # lag classes for empirical variogram
PERCENTILE_MAXDIST = 45     # max distance percentile for lag range
PURE_NUGGET_STD    = 0.02   # std of indicator below which = pure nugget
                             # (lower than AUDPC version because max std ≈ 0.5)

# Kriging grid
GRID_STEPS    = 80
SMOOTH_SIGMA  = 0.5   # Gaussian smoothing (grid cells) for display only

DPI = 400


# =============================================================================
# 1. DATA LOADING
# =============================================================================

def lat_lon_to_metres(df, lat_col="Latitude", lon_col="Longitude"):
    """Convert lat/lon to local metric coordinates centred on lot centroid."""
    lat0    = df[lat_col].mean()
    lon0    = df[lon_col].mean()
    cos_lat = np.cos(np.radians(lat0))
    df      = df.copy()
    df["Northing"] = (df[lat_col] - lat0) * 111_320
    df["Easting"]  = (df[lon_col] - lon0) * 111_320 * cos_lat
    return df


def load_lot(xlsx_path, label, has_utm=True):
    """
    Load a lot from Excel.
    has_utm=True  → Northing/Easting already in metres.
    has_utm=False → Latitude/Longitude converted to metres.
    """
    df = pd.read_excel(xlsx_path)
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    if not has_utm:
        df = lat_lon_to_metres(df)
    print(f"  [{label}] {len(df)} plants loaded from {xlsx_path}")
    return df


# =============================================================================
# 2. INDICATOR VARIABLE COMPUTATION
# =============================================================================

def compute_indicators(df, period_bounds=PERIOD_BOUNDS,
                        threshold=INCIDENCE_THRESHOLD):
    """
    For each evaluation period, create a binary indicator column:
        I_pt = 1 if max(severity in period) >= threshold
        I_pt = 0 otherwise

    The indicator uses the MAXIMUM severity observed within the period
    to capture any moment of infection.

    Returns augmented DataFrame and list of indicator column names.
    """
    disease_cols = sorted(
        [c for c in df.columns if c.startswith("D") and c[1:].isdigit()],
        key=lambda x: int(x[1:])
    )

    indicator_cols = []
    for pname, lo, hi in period_bounds:
        cols = [c for c in disease_cols if lo <= int(c[1:]) <= hi]
        if len(cols) < 1:
            continue
        # Use max severity in the period window
        max_sev = df[cols].max(axis=1)
        col_name = f"IND_{pname}"
        df[col_name] = (max_sev >= threshold).astype(float)
        indicator_cols.append(col_name)

    # Summary
    for col in indicator_cols:
        pct = df[col].mean() * 100
        print(f"    {col}: incidence = {pct:.1f}%")

    return df, indicator_cols


# =============================================================================
# 3. INDICATOR VARIOGRAM
# =============================================================================
#
# The indicator variogram is computed identically to the standard variogram
# but applied to the binary variable Z ∈ {0,1}.
# Its sill is bounded by p*(1-p), where p = proportion of presences.
# =============================================================================

def empirical_variogram(coords, values,
                         n_lags=N_LAGS,
                         pct_maxdist=PERCENTILE_MAXDIST):
    """
    Method-of-moments estimator (Matheron 1963) applied to any variable,
    including binary indicator variables.
    """
    dists    = cdist(coords, coords)
    vals     = np.asarray(values, dtype=float)
    max_dist = np.percentile(dists[dists > 0], pct_maxdist)
    lag_size = max_dist / n_lags
    lags, gammas, counts = [], [], []
    for i in range(n_lags):
        lo, hi = i * lag_size, (i + 1) * lag_size
        mask   = (dists > lo) & (dists <= hi)
        pi, pj = np.where(mask)
        if len(pi) > 10:
            lags.append((lo + hi) / 2)
            gammas.append(((vals[pi] - vals[pj])**2).mean() / 2)
            counts.append(len(pi))
    return np.array(lags), np.array(gammas), np.array(counts)


# =============================================================================
# 4. THEORETICAL VARIOGRAM MODELS
# =============================================================================

def model_spherical(h, nugget, sill, rang):
    return np.where(
        h <= rang,
        nugget + (sill - nugget) * (1.5*(h/rang) - 0.5*(h/rang)**3),
        sill,
    )

def model_exponential(h, nugget, sill, rang):
    return nugget + (sill - nugget) * (1 - np.exp(-h / rang))

def model_gaussian(h, nugget, sill, rang):
    return nugget + (sill - nugget) * (1 - np.exp(-(h / rang)**2))

MODELS = {
    "Spherical":   model_spherical,
    "Exponential": model_exponential,
    "Gaussian":    model_gaussian,
}

PYKRIGE_MODEL = {
    "Spherical":   "spherical",
    "Exponential": "exponential",
    "Gaussian":    "gaussian",
}


def fit_variogram_models(lags, gammas, p_hat):
    """
    Fit all three models. The theoretical sill is constrained to at most
    p*(1-p) for the indicator variogram.

    Parameters
    ----------
    lags, gammas : empirical variogram arrays
    p_hat        : proportion of presences (bounds the sill)

    Returns best_name, best_params, best_r2, all_results
    """
    theoretical_sill = p_hat * (1 - p_hat)
    sill_ub = max(gammas.max(), theoretical_sill) * 1.5
    s0 = min(gammas.max(), theoretical_sill)
    r0 = lags.max() * 0.5
    n0 = gammas.min() * 0.05

    all_results = {}
    for name, func in MODELS.items():
        try:
            popt, _ = curve_fit(
                func, lags, gammas,
                p0=[n0, s0, r0],
                bounds=([0, 0, 0], [sill_ub, sill_ub * 2, lags.max() * 2]),
                maxfev=8000,
            )
            pred   = func(lags, *popt)
            ss_res = np.sum((gammas - pred)**2)
            ss_tot = np.sum((gammas - gammas.mean())**2)
            r2     = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            rmse   = np.sqrt(ss_res / len(lags))
            all_results[name] = dict(params=popt, r2=r2, rmse=rmse)
        except Exception:
            all_results[name] = None

    valid = {k: v for k, v in all_results.items() if v is not None}
    if not valid:
        return None, None, None, all_results
    best_name = max(valid, key=lambda k: valid[k]["r2"])
    p         = valid[best_name]
    return best_name, p["params"], p["r2"], all_results


def run_variogram_analysis(df, indicator_cols, label):
    """
    Compute indicator variograms and fit theoretical models for all periods.
    Returns vario_results and params_rows for CSV export.
    """
    coords        = df[["Northing", "Easting"]].values
    vario_results = []
    params_rows   = []

    for idx, col in enumerate(indicator_cols):
        period  = idx + 1
        z       = df[col].fillna(0).values.astype(float)
        p_hat   = z.mean()
        std_val = z.std()

        print(f"  [{label}] P{period}  p={p_hat:.3f}  "
              f"p*(1-p)={p_hat*(1-p_hat):.4f}  std={std_val:.4f}", end="")

        if std_val < PURE_NUGGET_STD:
            print(" → pure nugget")
            vario_results.append(dict(period=period, pure_nugget=True,
                                      best_name=None, params=None,
                                      r2=None, p_hat=p_hat))
            params_rows.append(dict(Lot=label, Period=f"P{period}",
                                    Model="Pure nugget", p_hat=round(p_hat,4),
                                    Nugget=np.nan, Sill=np.nan, Range_m=np.nan,
                                    R2=np.nan, RMSE=np.nan, CI_pct=np.nan))
            continue

        lags, gammas, counts = empirical_variogram(coords, z)
        if len(lags) < 3:
            print(" → too few lag classes")
            continue

        best_name, params, r2, all_res = fit_variogram_models(
            lags, gammas, p_hat)
        if best_name is None:
            print(" → no model converged")
            continue

        nu, si, ra = params
        ci = (nu / si * 100) if si > 0 else 0
        print(f" → best={best_name}  R²={r2:.3f}  "
              f"nugget={nu:.4f}  sill={si:.4f}  range={ra:.1f} m")

        vario_results.append(dict(
            period=period, pure_nugget=False,
            lags=lags, gammas=gammas, counts=counts,
            best_name=best_name, params=params,
            r2=r2, all_results=all_res, p_hat=p_hat,
        ))

        for mname, mres in all_res.items():
            if mres:
                nu_m, si_m, ra_m = mres["params"]
                ci_m = (nu_m / si_m * 100) if si_m > 0 else 0
                params_rows.append(dict(
                    Lot=label, Period=f"P{period}", Model=mname,
                    p_hat=round(p_hat, 4),
                    Nugget=round(nu_m, 5), Sill=round(si_m, 5),
                    Range_m=round(ra_m, 1),
                    R2=round(mres["r2"], 4), RMSE=round(mres["rmse"], 5),
                    CI_pct=round(ci_m, 1),
                    Best="*" if mname == best_name else "",
                ))

    return vario_results, params_rows


# =============================================================================
# 5. INDICATOR KRIGING INTERPOLATION
# =============================================================================

def hull_mask(xi, yi, x_obs, y_obs):
    """Boolean mask — True inside convex hull of observation points."""
    pts = np.column_stack([x_obs, y_obs])
    try:
        hull = ConvexHull(pts)
        tri  = Delaunay(pts[hull.vertices])
        XI, YI = np.meshgrid(xi, yi)
        inside = tri.find_simplex(
            np.column_stack([XI.ravel(), YI.ravel()])
        ) >= 0
        return inside.reshape(XI.shape)
    except Exception:
        return np.ones((len(yi), len(xi)), dtype=bool)


def run_indicator_kriging(df, vario_results, indicator_cols,
                           grid_steps=GRID_STEPS, label=""):
    """
    Ordinary Kriging applied to the binary indicator variable.
    The output is an estimated probability of disease presence P(Z=1)
    at each grid node, clipped to [0, 1].

    Falls back to inverse-distance weighting (IDW) on failure.

    Returns list of per-period dicts:
        period, prob_grid, xi, yi, x_obs, y_obs, z_obs, p_hat
    """
    x_raw = df["Easting"].values.astype(float)
    y_raw = df["Northing"].values.astype(float)
    x_min, x_max = x_raw.min(), x_raw.max()
    y_min, y_max = y_raw.min(), y_raw.max()
    span = max(x_max - x_min, y_max - y_min)

    x_n  = (x_raw - x_min) / span
    y_n  = (y_raw - y_min) / span
    xi_n = np.linspace(0, (x_max - x_min) / span, grid_steps)
    yi_n = np.linspace(0, (y_max - y_min) / span, grid_steps)

    vd_map  = {v["period"]: v for v in vario_results}
    results = []

    for idx, col in enumerate(indicator_cols):
        period = idx + 1
        z      = df[col].fillna(0).values.astype(float)
        p_hat  = z.mean()
        vd     = vd_map.get(period, {})
        prob_grid = None

        # ── Indicator Kriging ─────────────────────────────────────────────────
        if (not vd.get("pure_nugget", False)
                and vd.get("best_name") is not None):
            nu, si, ra = vd["params"]
            pykrige_m  = PYKRIGE_MODEL[vd["best_name"]]
            psill      = max(si - nu, 1e-6)
            rang       = max(ra / span, 1e-5)
            nug        = max(nu, 0.0)
            try:
                OK = OrdinaryKriging(
                    x_n, y_n, z,
                    variogram_model=pykrige_m,
                    variogram_parameters=[psill, rang, nug],
                    verbose=False, enable_plotting=False,
                )
                raw_grid, _ = OK.execute("grid", xi_n, yi_n)
                # Post-processing: clip to valid probability range [0, 1]
                prob_grid   = np.clip(np.array(raw_grid), 0.0, 1.0)
            except Exception as e:
                print(f"  [{label}] P{period}: IK failed ({e}) → IDW fallback")

        # ── IDW fallback (preserves indicator nature) ─────────────────────────
        if prob_grid is None:
            XX, YY = np.meshgrid(xi_n, yi_n)
            prob_grid = np.zeros_like(XX, dtype=float)
            for i in range(grid_steps):
                for j in range(grid_steps):
                    d2 = (x_n - XX[i, j])**2 + (y_n - YY[i, j])**2
                    d2 = np.maximum(d2, 1e-10)
                    prob_grid[i, j] = np.clip(
                        np.sum(z / d2) / np.sum(1.0 / d2), 0.0, 1.0)

        results.append(dict(
            period=period, col=col,
            prob_grid=prob_grid,
            xi=xi_n * (x_max - x_min) + x_min,
            yi=yi_n * (y_max - y_min) + y_min,
            x_obs=x_raw, y_obs=y_raw, z_obs=z,
            p_hat=p_hat,
        ))
        print(f"  [{label}] P{period}: p̂={p_hat:.3f}  "
              f"grid_range=[{prob_grid.min():.3f}, {prob_grid.max():.3f}]")

    return results


# =============================================================================
# 6. GLOBAL PLOT STYLE
# =============================================================================

plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":         9,
    "axes.facecolor":    "white",
    "figure.facecolor":  "white",
    "axes.grid":         True,
    "grid.color":        "#DDDDDD",
    "grid.linewidth":    0.5,
    "grid.linestyle":    "-",
    "axes.linewidth":    0.5,
    "xtick.major.size":  2.5, "ytick.major.size":  2.5,
    "xtick.major.width": 0.5, "ytick.major.width": 0.5,
    "xtick.direction":   "out", "ytick.direction":  "out",
    "pdf.fonttype":      42,   "ps.fonttype":       42,
})

BLACK      = "#000000"
DGRAY      = "#333333"
LGRAY      = "#CCCCCC"
PANEL_BG   = "#FAFAFA"

FS_PANEL   = 18
FS_TITLE   = 11
FS_LABEL   = 10
FS_TICK    =  9
FS_ANNOT   =  9.5
FS_LEG     = 10
AW         =  8

PANEL_LBL  = list("ABCDEFG")
POSITIONS  = [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2),(2,1)]  # 3+3+1

# ── Colormaps ─────────────────────────────────────────────────────────────────
# Probability map: white (0) → yellow → orange → deep red (1)
CMAP_PROB = LinearSegmentedColormap.from_list(
    "ik_prob",
    ["#FFFFFF","#FFF3B0","#FFD166","#EF8C00","#C0392B","#7B0000"],
    N=512,
)
# Indicator points: white (absent) / deep color (present)
CMAP_IND  = LinearSegmentedColormap.from_list(
    "ik_ind", ["#EEEEEE","#1B4F8A"], N=2)

LEGEND_HANDLES = [
    Line2D([0],[0], color=BLACK, lw=1.6,   label="Fitted curve"),
    Line2D([0],[0], marker="^", color=DGRAY, lw=0, ms=7,
           label="Adjusted values"),
    Line2D([0],[0], marker="o", color="white", lw=0, ms=7,
           markeredgecolor=BLACK, markeredgewidth=1.2,
           label="Empirical values"),
]


def _save(fig, basename):
    for ext in ("png", "pdf"):
        path = os.path.join(OUTPUT_DIR, f"{basename}.{ext}")
        fig.savefig(path, dpi=DPI if ext == "png" else None,
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {basename}.png / .pdf")


# =============================================================================
# 7. FIGURE A — INDICATOR VARIOGRAMS
# =============================================================================

def figure_indicator_variograms(vario_results, df, indicator_cols, lot_label):
    """
    2-row × 4-column grid of indicator semivariograms (7 periods + legend).
    The theoretical sill p*(1-p) is drawn as a horizontal reference line.
    """
    vd_map = {vd["period"]: vd for vd in vario_results}

    fig = plt.figure(figsize=(18, 10), facecolor="white", dpi=DPI)
    gs  = gridspec.GridSpec(2, 4, figure=fig,
                             hspace=0.40, wspace=0.28,
                             left=0.065, right=0.975,
                             top=0.920,  bottom=0.090)

    for idx in range(7):
        period = idx + 1
        r, c   = divmod(idx, 4)
        ax     = fig.add_subplot(gs[r, c])
        vd     = vd_map.get(period)
        col    = indicator_cols[idx] if idx < len(indicator_cols) else None
        ax.set_facecolor(PANEL_BG)

        ds = (period - 1) * 180
        de = min(period * 180, 1200)
        ax.set_title(f"Period {period}  (D{ds}–D{de})",
                     fontsize=FS_TITLE, fontweight="bold", pad=4, color=BLACK)

        if vd and not vd.get("pure_nugget", False) and vd.get("best_name"):
            lags   = vd["lags"]; gammas = vd["gammas"]
            best   = vd["best_name"]
            params = vd["params"]
            nu, si, ra = params
            r2  = vd["all_results"][best]["r2"]
            ci  = (nu / si * 100) if si > 0 else 0
            p   = vd["p_hat"]
            pq  = p * (1 - p)   # theoretical sill

            # Theoretical sill reference
            ax.axhline(pq, color="#1B4F8A", lw=0.8, ls="-.",
                       alpha=0.6, zorder=1,
                       label=f"p(1-p) = {pq:.4f}")
            # Fitted sill
            ax.axhline(si, color=LGRAY, lw=0.7, ls="--", zorder=1)
            ax.axvline(ra, color=LGRAY, lw=0.7, ls="--", zorder=1)
            ax.axhline(nu, color=LGRAY, lw=0.6, ls=":", zorder=1)
            ax.plot(0, nu, "o", ms=4.5, color="white",
                    markeredgecolor=BLACK, markeredgewidth=1.0,
                    zorder=6, clip_on=False)

            h_fit = np.linspace(lags[0]*0.3, lags.max()*1.04, 500)
            ax.plot(h_fit, MODELS[best](h_fit, *params),
                    color=BLACK, lw=1.6, zorder=3)
            ax.scatter(lags, MODELS[best](lags, *params),
                       marker="^", s=30, color=DGRAY, zorder=4, linewidths=0)
            ax.scatter(lags, gammas, s=34, color="white",
                       edgecolors=BLACK, linewidths=1.1, zorder=5)

            rows = [
                ("Nugget", f"{nu:>{AW}.5f}"),
                ("Sill",   f"{si:>{AW}.5f}"),
                ("p(1-p)", f"{pq:>{AW}.5f}"),
                ("Range",  f"{ra:>{AW}.1f} m"),
                ("R²",     f"{r2:>{AW}.4f}"),
                ("CI",     f"{ci:>{AW}.1f} %"),
                ("Model",  f"{best:>{AW}}"),
            ]
            kw  = max(len(k) for k, _ in rows)
            txt = "\n".join(f"{k:<{kw}} = {v}" for k, v in rows)
            ax.text(0.975, 0.035, txt, transform=ax.transAxes,
                    fontsize=FS_ANNOT - 0.5, va="bottom", ha="right",
                    fontfamily="monospace", fontstyle="normal", linespacing=1.45,
                    bbox=dict(boxstyle="round,pad=0.42", fc="white",
                              ec=LGRAY, lw=0.7, alpha=0.96))
        else:
            ax.text(0.5, 0.5, "Pure nugget\n(no spatial structure)",
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=FS_ANNOT, color=DGRAY,
                    bbox=dict(boxstyle="round,pad=0.5", fc="white",
                              ec=LGRAY, lw=0.7))

        ax.set_xlabel("Distance (m)", fontsize=FS_LABEL, labelpad=3)
        if c == 0:
            ax.set_ylabel("Indicator semivariance γ(h)",
                          fontsize=FS_LABEL, labelpad=3)
        ax.tick_params(labelsize=FS_TICK, pad=2)
        ax.set_xlim(left=0); ax.set_ylim(bottom=0)
        ax.yaxis.set_major_locator(mticker.MaxNLocator(4))
        ax.xaxis.set_major_locator(mticker.MaxNLocator(5, integer=True))

    # Legend panel (slot [1,3])
    ax_leg = fig.add_subplot(gs[1, 3])
    ax_leg.set_visible(False)
    pos = ax_leg.get_position()
    extra_handles = LEGEND_HANDLES + [
        Line2D([0],[0], color="#1B4F8A", lw=0.8, ls="-.",
               alpha=0.6, label="Theoretical sill p(1-p)"),
    ]
    leg = fig.legend(
        handles=extra_handles, fontsize=FS_LEG, loc="center",
        bbox_to_anchor=(pos.x0 + pos.width/2, pos.y0 + pos.height/2),
        bbox_transform=fig.transFigure,
        framealpha=0.95, edgecolor=LGRAY, fancybox=False,
        borderpad=0.9, handlelength=1.8, handletextpad=0.6,
        title="Legend", title_fontsize=FS_LEG,
    )
    leg.get_title().set_fontweight("bold")

    fig.suptitle(
        f"Indicator variograms of PRR incidence (Z = 1 if severity ≥ {INCIDENCE_THRESHOLD})"
        f"  —  Lot: {lot_label}",
        fontsize=12, fontweight="bold", y=0.998, color=BLACK,
    )
    _save(fig, f"Fig_IK_Variogram_{lot_label.replace(' ','_')}")


# =============================================================================
# 8. FIGURE B — KRIGED PROBABILITY MAPS  (3+3+1 layout, viridis style)
# =============================================================================

def _discrete_legend(ax, vmin, vmax, cmap, n_bins=5, title="P(presence)"):
    bounds  = np.linspace(vmin, vmax, n_bins + 1)
    labels  = [f"{bounds[i]:.2f} – {bounds[i+1]:.2f}" for i in range(n_bins)]
    colors  = [cmap(i / (n_bins - 1)) for i in range(n_bins)]
    patches = [mpatches.Patch(facecolor=colors[i], edgecolor="none",
                               label=labels[i])
               for i in range(n_bins - 1, -1, -1)]
    leg = ax.legend(handles=patches, title=title,
                    loc="lower right", fontsize=6.5, title_fontsize=7,
                    framealpha=0.92, edgecolor="#AAAAAA", fancybox=False,
                    borderpad=0.6, handlelength=1.2, handletextpad=0.5)
    leg.get_title().set_fontweight("bold")


def figure_probability_maps(ik_results, lot_label):
    """
    7-panel (3+3+1) figure of kriged P(Z=1) maps.
    White background, viridis colormap, convex hull masking.
    """
    norm = Normalize(vmin=0.0, vmax=1.0)
    CMAP = plt.cm.viridis

    fig = plt.figure(figsize=(11, 14), facecolor="white", dpi=DPI)
    gs  = gridspec.GridSpec(3, 3, figure=fig,
                             hspace=0.18, wspace=0.12,
                             left=0.10, right=0.95,
                             top=0.93, bottom=0.06)

    for pidx, kdat in enumerate(ik_results):
        r, c    = POSITIONS[pidx]
        ax      = fig.add_subplot(gs[r, c])
        period  = kdat["period"]
        xi, yi  = kdat["xi"], kdat["yi"]
        pg      = gaussian_filter(kdat["prob_grid"], sigma=SMOOTH_SIGMA)
        pg      = np.clip(pg, 0.0, 1.0)
        xo, yo, zo = kdat["x_obs"], kdat["y_obs"], kdat["z_obs"]

        # Convex hull mask
        mask = hull_mask(xi, yi, xo, yo)
        pg_m = np.where(mask, pg, np.nan)

        # Probability surface
        ax.imshow(pg_m, origin="lower",
                  extent=[xi.min(), xi.max(), yi.min(), yi.max()],
                  cmap=CMAP, norm=norm, aspect="auto",
                  interpolation="bilinear", zorder=2)

        ax.set_axisbelow(True)
        ax.grid(True, color="#DDDDDD", lw=0.5, zorder=1)

        # Observation points (presence/absence)
        ax.scatter(xo[zo==0], yo[zo==0], c="#CCCCCC", s=1.0,
                   linewidths=0, zorder=3, alpha=0.5)
        ax.scatter(xo[zo==1], yo[zo==1], c="black", s=1.5,
                   linewidths=0, zorder=4, alpha=0.6)

        # Panel letter
        ax.text(0.03, 0.97, PANEL_LBL[pidx],
                transform=ax.transAxes, fontsize=10,
                fontweight="bold", color=BLACK, ha="left", va="top", zorder=5)

        # Period + prevalence label
        ds = (period - 1) * 180
        de = min(period * 180, 1200)
        p  = kdat["p_hat"]
        ax.text(0.97, 0.97, f"D{ds}–D{de}\np̂={p:.1%}",
                transform=ax.transAxes, fontsize=6.0, color=BLACK,
                ha="right", va="top", linespacing=1.4,
                bbox=dict(boxstyle="round,pad=0.25",
                          fc="white", ec="none", alpha=0.75))

        ax.tick_params(labelsize=6.5, colors=BLACK, pad=2)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(5, integer=True))

        if c == 0:
            ax.set_ylabel("Spatial dimension in Y",
                          fontsize=7.5, labelpad=3, color=BLACK)
        if r == 2 or (r == 1):
            ax.set_xlabel("Spatial dimension in X",
                          fontsize=7.5, labelpad=3, color=BLACK)

        ax.set_facecolor("white")
        for sp in ax.spines.values():
            sp.set_color("#BBBBBB"); sp.set_linewidth(0.5)

        # Discrete legend on last panel only
        if pidx == 6:
            _discrete_legend(ax, 0, 1, CMAP, n_bins=5,
                             title="P(disease presence)")

    # Hide unused cells
    for c_hide in [0, 2]:
        fig.add_subplot(gs[2, c_hide]).set_visible(False)

    fig.suptitle(
        f"Indicator kriging — estimated probability of PRR presence\n"
        f"Lot: {lot_label}",
        fontsize=11, fontweight="bold", y=0.97,
        color=BLACK, linespacing=1.35,
    )
    _save(fig, f"Fig_IK_Maps_{lot_label.replace(' ','_')}")


# =============================================================================
# 9. FIGURE C — OBSERVED INDICATOR MAPS
# =============================================================================

def figure_indicator_maps(df, indicator_cols, lot_label):
    """
    7-panel (3+3+1) scatter map of the observed binary indicator per period.
    Absent plants: light grey.  Present plants: dark blue.
    """
    fig = plt.figure(figsize=(11, 14), facecolor="white", dpi=DPI)
    gs  = gridspec.GridSpec(3, 3, figure=fig,
                             hspace=0.18, wspace=0.12,
                             left=0.10, right=0.95,
                             top=0.93, bottom=0.06)

    pt = max(1.5, min(5, 30_000 / len(df)))
    x  = df["Easting"].values
    y  = df["Northing"].values

    for pidx, col in enumerate(indicator_cols):
        r, c   = POSITIONS[pidx]
        ax     = fig.add_subplot(gs[r, c])
        z      = df[col].fillna(0).values
        period = pidx + 1
        p_hat  = z.mean()

        ax.set_axisbelow(True)
        ax.grid(True, color="#DDDDDD", lw=0.5, zorder=1)
        ax.scatter(x[z==0], y[z==0], c="#DDDDDD", s=pt,
                   linewidths=0, zorder=2, alpha=0.7, label="Absent (0)")
        ax.scatter(x[z==1], y[z==1], c="#1B4F8A", s=pt,
                   linewidths=0, zorder=3, alpha=0.85, label="Present (1)")

        ax.text(0.03, 0.97, PANEL_LBL[pidx],
                transform=ax.transAxes, fontsize=10,
                fontweight="bold", color=BLACK, ha="left", va="top", zorder=5)

        ds = (period - 1) * 180
        de = min(period * 180, 1200)
        ax.text(0.97, 0.97, f"D{ds}–D{de}\np̂={p_hat:.1%}",
                transform=ax.transAxes, fontsize=6.0, color=BLACK,
                ha="right", va="top", linespacing=1.4,
                bbox=dict(boxstyle="round,pad=0.25",
                          fc="white", ec="none", alpha=0.75))

        ax.tick_params(labelsize=6.5, colors=BLACK, pad=2)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(5, integer=True))
        if c == 0:
            ax.set_ylabel("Northing (m)", fontsize=7.5, labelpad=3)
        if r == 2 or r == 1:
            ax.set_xlabel("Easting (m)", fontsize=7.5, labelpad=3)
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_facecolor("white")
        for sp in ax.spines.values():
            sp.set_color("#BBBBBB"); sp.set_linewidth(0.5)

        # Legend on last panel
        if pidx == 6:
            leg = ax.legend(fontsize=7, loc="lower right",
                            framealpha=0.9, edgecolor="#AAAAAA",
                            fancybox=False, markerscale=2.5)

    for c_hide in [0, 2]:
        fig.add_subplot(gs[2, c_hide]).set_visible(False)

    fig.suptitle(
        f"Observed PRR incidence indicator (Z = 1 if severity ≥ {INCIDENCE_THRESHOLD})\n"
        f"Lot: {lot_label}",
        fontsize=11, fontweight="bold", y=0.97,
        color=BLACK, linespacing=1.35,
    )
    _save(fig, f"Fig_IK_Points_{lot_label.replace(' ','_')}")


# =============================================================================
# 10. MODEL COMPARISON TABLE
# =============================================================================

PERIOD_COLORS = {
    "P1":"#F0F0F0","P2":"white","P3":"#F0F0F0",
    "P4":"white",  "P5":"#F0F0F0","P6":"white","P7":"#F0F0F0",
}

def figure_model_table(params_rows, lot_label):
    """Render indicator variogram model comparison table."""
    df_res = pd.DataFrame(params_rows)
    cols   = ["Period","Model","p_hat","Nugget","Sill","Range_m",
              "R2","RMSE","CI_pct","Best"]
    col_labels = ["Period","Model","p̂","Nugget","Sill","Range (m)",
                  "R²","RMSE","CI (%)","Best"]
    df_res = df_res[cols] if all(c in df_res.columns for c in cols) else df_res
    data   = df_res.values.tolist()

    fig, ax = plt.subplots(figsize=(17, 8), facecolor="white")
    ax.axis("off")
    tbl = ax.table(cellText=data, colLabels=col_labels,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1.0, 1.85)

    for (r, c_), cell in tbl.get_celld().items():
        cell.set_edgecolor("#AAAAAA"); cell.set_linewidth(0.5)
        if r == 0:
            cell.set_facecolor("#1F3864")
            cell.set_text_props(color="white", fontweight="bold", fontsize=9.5)
        else:
            row_d = data[r - 1]
            base  = PERIOD_COLORS.get(str(row_d[0]), "white")
            if str(row_d[-1]) == "*":
                cell.set_facecolor("#D0D0D0")
                cell.set_text_props(fontweight="bold")
            else:
                cell.set_facecolor(base)

    for r in range(1, len(data) + 1):
        if str(data[r-1][-1]) == "*":
            tbl[r, len(cols)-1].get_text().set_text("Best")
            tbl[r, len(cols)-1].get_text().set_fontweight("bold")

    ax.set_title(
        f"Indicator variogram model fit comparison  —  {lot_label}\n"
        "(Nugget, Sill, Range, R², RMSE, Cambardella index; "
        "shaded = best-fit model per period)",
        fontsize=11, fontweight="bold", pad=14, color="black",
    )
    plt.tight_layout()
    _save(fig, f"Table_IK_{lot_label.replace(' ','_')}")


# =============================================================================
# 11. MAIN PIPELINE
# =============================================================================

if __name__ == "__main__":

    lots_config = [
        (DONMATIAS_XLSX, "Donmatias", True),    # has_utm=True
        (EL_RETIRO_XLSX, "El Retiro", False),   # has_utm=False (Lat/Lon)
        (LA_CEJA_XLSX,   "La Ceja",   False),
    ]

    for xlsx, label, has_utm in lots_config:

        if xlsx is None or not os.path.exists(xlsx):
            print(f"\n[{label}] file not found — skipped.")
            continue

        print(f"\n{'='*60}")
        print(f"  LOT: {label}")
        print(f"{'='*60}")

        # Step 1: load data
        df = load_lot(xlsx, label, has_utm=has_utm)

        # Step 2: compute binary indicator per period
        print("\n  Indicator variable (incidence):")
        df, indicator_cols = compute_indicators(df)
        print(f"  Indicator columns: {indicator_cols}")

        # Step 3: indicator variogram analysis
        print("\n  Indicator variogram fitting:")
        vario_results, params_rows = run_variogram_analysis(
            df, indicator_cols, label)

        # Save parameters CSV
        csv_path = os.path.join(
            OUTPUT_DIR,
            f"indicator_variogram_params_{label.replace(' ','_')}.csv"
        )
        pd.DataFrame(params_rows).to_csv(csv_path, index=False)
        print(f"  Parameters saved: {csv_path}")

        # Step 4: indicator kriging
        print("\n  Indicator kriging:")
        ik_results = run_indicator_kriging(
            df, vario_results, indicator_cols, label=label)

        # Step 5: figures
        print("\n  Generating figures:")
        figure_indicator_variograms(
            vario_results, df, indicator_cols, label)
        figure_probability_maps(ik_results, label)
        figure_indicator_maps(df, indicator_cols, label)
        figure_model_table(params_rows, label)

    print("\nAll analyses complete.")

In [ ]:
"""
Geostatistical Spatial Analysis of PRR Severity
================================================
Reproduces the full analysis pipeline for three lots:
  - Donmatias  : read from Taller_epi.xlsx
  - El Retiro  : read from Data_ElRetiro.xlsx
  - La Ceja    : read from Data_LaCeja.xlsx

Outputs per lot (saved to OUTPUT_DIR):
  Fig_A_<lot>.png / .pdf   -> Empirical + fitted semivariograms (7 periods)
  Fig_B_<lot>.png / .pdf   -> Spatial distribution maps        (7 periods)
  Table_<lot>.png / .pdf   -> Variogram model comparison table

Dependencies
------------
    pip install pandas numpy scipy matplotlib openpyxl

Usage
-----
    python geostatistical_analysis_3lots.py
"""

import os
import warnings
import numpy  as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.optimize         import curve_fit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
from matplotlib.colors import Normalize
from matplotlib.cm     import ScalarMappable
from matplotlib.lines  import Line2D
warnings.filterwarnings("ignore")

# =============================================================================
# 0. USER CONFIGURATION
# =============================================================================

# Input files — adjust paths as needed
DONMATIAS_XLSX = "Taller_epi.xlsx"
EL_RETIRO_XLSX = "Data_ElRetiro_1338.xlsx"  #Data not included due to confidentiality
LA_CEJA_XLSX   = "Data_LaCeja_1666.xlsx"    #Data not included due to confidentiality

OUTPUT_DIR = "outputs"                       # folder for all outputs
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Coordinate columns: set to ('Latitude','Longitude') if UTM not available
COORD_COLS_DONMATIAS = ("Northing", "Easting")   # already in metres
COORD_COLS_OTHERS    = ("Latitude", "Longitude")  # will be converted to metres

# Evaluation periods (days) — must match column names in the data
PERIOD_BOUNDS = [
    ("P1",   0, 180),
    ("P2", 180, 360),
    ("P3", 360, 540),
    ("P4", 540, 720),
    ("P5", 720, 900),
    ("P6", 900, 1080),
    ("P7",1080, 1200),
]

N_LAGS            = 12      # number of lag classes for empirical variogram
PERCENTILE_MAXDIST = 45     # % of all pairwise distances used as max lag
PURE_NUGGET_STD    = 0.05   # if std < this, flag as pure nugget (no structure)

# Figure settings
DPI_FIGURE = 400
MAP_COLORMAP = plt.cm.YlOrRd


# =============================================================================
# 1. DATA LOADING
# =============================================================================

def lat_lon_to_metres(df, lat_col="Latitude", lon_col="Longitude"):
    """
    Convert lat/lon columns to local metric coordinates (metres) centred
    on the lot centroid.  Returns df with new columns 'Northing', 'Easting'.
    """
    lat0 = df[lat_col].mean()
    lon0 = df[lon_col].mean()
    cos_lat = np.cos(np.radians(lat0))
    df = df.copy()
    df["Northing"] = (df[lat_col] - lat0) * 111_320
    df["Easting"]  = (df[lon_col] - lon0) * 111_320 * cos_lat
    return df


def load_lot(xlsx_path, coord_cols, label):
    """
    Load a lot from an Excel file.  Returns a tidy DataFrame with
    columns: GPS_Height, Vert_Prec, Horz_Prec, Northing, Easting,
             Point_ID, ORIG_FID, D0, D60, …, D1200
    """
    df = pd.read_excel(xlsx_path)
    df.columns = [c.strip() for c in df.columns]

    if coord_cols == ("Latitude", "Longitude"):
        df = lat_lon_to_metres(df)

    # Normalise disease column names: 'D 60' -> 'D60'
    rename = {c: c.replace(" ", "") for c in df.columns if c.startswith("D")}
    df = df.rename(columns=rename)

    print(f"  [{label}]  {len(df)} plants loaded from {xlsx_path}")
    return df


# =============================================================================
# 2. AUDPC COMPUTATION
# =============================================================================

def compute_audpc(df, period_bounds):
    """
    Add AUDPC columns (trapezoidal rule) and their square roots for each
    evaluation period.  Returns the augmented DataFrame and the list of
    sqrt-AUDPC column names.
    """
    disease_cols = sorted(
        [c for c in df.columns if c.startswith("D") and c[1:].isdigit()],
        key=lambda x: int(x[1:])
    )

    for pname, lo, hi in period_bounds:
        cols  = [c for c in disease_cols if lo <= int(c[1:]) <= hi]
        if len(cols) < 2:
            continue
        times = [int(c[1:]) for c in cols]
        vals  = df[cols].values
        df[f"AUDPC_{pname}"]      = np.trapezoid(vals, times, axis=1)
        df[f"AUDPC_{pname}_sqrt"] = np.sqrt(np.maximum(df[f"AUDPC_{pname}"], 0))

    audpc_cols = [c for c in df.columns if "AUDPC_P" in c and "sqrt" in c]
    return df, audpc_cols


# =============================================================================
# 3. VARIOGRAM ANALYSIS
# =============================================================================

# --- Theoretical model functions -------------------------------------------

def model_spherical(h, nugget, sill, rang):
    return np.where(
        h <= rang,
        nugget + (sill - nugget) * (1.5*(h/rang) - 0.5*(h/rang)**3),
        sill
    )

def model_exponential(h, nugget, sill, rang):
    return nugget + (sill - nugget) * (1 - np.exp(-h / rang))

def model_gaussian(h, nugget, sill, rang):
    return nugget + (sill - nugget) * (1 - np.exp(-(h / rang)**2))

MODELS = {
    "Spherical":   model_spherical,
    "Exponential": model_exponential,
    "Gaussian":    model_gaussian,
}


def empirical_variogram(coords, values, n_lags=N_LAGS,
                         pct_maxdist=PERCENTILE_MAXDIST):
    """
    Compute the empirical (experimental) variogram using the
    method-of-moments estimator (Matheron 1963).
    """
    dists    = cdist(coords, coords)
    vals     = np.asarray(values, dtype=float)
    max_dist = np.percentile(dists[dists > 0], pct_maxdist)
    lag_size = max_dist / n_lags

    lags, gammas, counts = [], [], []
    for i in range(n_lags):
        lo, hi = i * lag_size, (i + 1) * lag_size
        mask   = (dists > lo) & (dists <= hi)
        pi, pj = np.where(mask)
        if len(pi) > 10:
            lags.append((lo + hi) / 2)
            gammas.append(((vals[pi] - vals[pj])**2).mean() / 2)
            counts.append(len(pi))

    return np.array(lags), np.array(gammas), np.array(counts)


def fit_all_models(lags, gammas):
    """
    Fit all theoretical models to the empirical variogram via non-linear
    least squares (leave-one-out cross-validation proxy: minimise RSS).
    Returns a dict {model_name: {'params', 'r2', 'rmse'} | None}.
    """
    results = {}
    s0 = gammas.max(); r0 = lags.max() * 0.5; n0 = gammas.min() * 0.1

    for name, func in MODELS.items():
        try:
            popt, _ = curve_fit(
                func, lags, gammas,
                p0=[n0, s0, r0],
                bounds=([0, 0, 0], [s0*2, s0*3, lags.max()*2]),
                maxfev=5000
            )
            pred   = func(lags, *popt)
            ss_res = np.sum((gammas - pred)**2)
            ss_tot = np.sum((gammas - gammas.mean())**2)
            r2     = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            rmse   = np.sqrt(ss_res / len(lags))
            results[name] = {"params": popt, "r2": r2, "rmse": rmse}
        except Exception:
            results[name] = None

    return results


def morisita_index(values):
    """
    Morisita (1962) aggregation index.
    Im > 1 → aggregated; Im = 1 → random; Im < 1 → uniform.
    """
    counts = np.round(values).astype(int)
    n = len(counts)
    S = counts.sum()
    if S == 0:
        return np.nan
    return n * np.sum(counts * (counts - 1)) / (S * (S - 1))


def analyse_lot(df, audpc_cols, label):
    """
    Run the full variogram analysis for all periods of one lot.
    Returns:
        vario_data   – list of per-period result dicts
        all_results  – list of per-model result dicts (for comparison table)
    """
    coords     = df[["Northing", "Easting"]].values
    vario_data = []
    all_results = []

    for idx, col in enumerate(audpc_cols):
        vals      = df[col].fillna(0).values
        period    = idx + 1
        std_val   = vals.std()

        print(f"  [{label}] P{period}  std={std_val:.3f}", end="")

        # Pure nugget: no spatial structure detectable
        if std_val < PURE_NUGGET_STD:
            print(" → pure nugget")
            vario_data.append({"period": period, "pure_nugget": True,
                               "morisita": np.nan})
            continue

        lags, gammas, counts = empirical_variogram(coords, vals)
        if len(lags) < 3:
            print(" → insufficient lag classes")
            continue

        model_fits = fit_all_models(lags, gammas)
        valid      = {k: v for k, v in model_fits.items() if v}
        if not valid:
            print(" → no model converged")
            continue

        best = max(valid, key=lambda k: valid[k]["r2"])
        Im   = morisita_index(vals)
        print(f" → best={best}  R²={valid[best]['r2']:.3f}  Im={Im:.3f}")

        vario_data.append({
            "period":     period,
            "lags":       lags,
            "gammas":     gammas,
            "counts":     counts,
            "model_fits": model_fits,
            "best_name":  best,
            "morisita":   Im,
            "pure_nugget": False,
        })

        for mname, mres in model_fits.items():
            if mres:
                nu, si, ra = mres["params"]
                ci = (nu / si * 100) if si > 0 else 0
                all_results.append({
                    "Lot":            label,
                    "Period":         f"P{period}",
                    "Model":          mname,
                    "Nugget":         round(nu, 2),
                    "Sill":           round(si, 2),
                    "Range (m)":      round(ra, 1),
                    "R²":             round(mres["r2"],   3),
                    "RMSE":           round(mres["rmse"],  3),
                    "Cambardella (%)":round(ci, 1),
                    "Best":           "*" if mname == best else "",
                })

    return vario_data, all_results


# =============================================================================
# 4. FIGURE HELPERS
# =============================================================================

# Global plot style
plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":         10,
    "axes.linewidth":    0.7,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "xtick.major.width": 0.7,
    "ytick.major.width": 0.7,
    "xtick.major.size":  3,
    "ytick.major.size":  3,
    "xtick.direction":   "out",
    "ytick.direction":   "out",
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

BLACK    = "#000000"
DGRAY    = "#333333"
LGRAY    = "#CCCCCC"
PANEL_BG = "#FAFAFA"
FS_PANEL = 18
FS_TITLE = 11
FS_LABEL = 10
FS_TICK  =  9
FS_ANNOT =  9.5
FS_LEG   = 10
AW       =  8       # annotation value column width (monospace alignment)

LEGEND_HANDLES = [
    Line2D([0],[0], color=BLACK, lw=1.6,   label="Fitted curve"),
    Line2D([0],[0], marker="^", color=DGRAY, lw=0, ms=7,
           label="Adjusted values"),
    Line2D([0],[0], marker="o", color="white", lw=0, ms=7,
           markeredgecolor=BLACK, markeredgewidth=1.2,
           label="Empirical values"),
]


def _ax_variogram(ax, vd, show_ylabel):
    """Draw a single semivariogram panel."""
    lags   = vd["lags"];  gammas = vd["gammas"]
    best   = vd["best_name"]
    params = vd["model_fits"][best]["params"]
    nu, si, ra = params
    r2  = vd["model_fits"][best]["r2"]
    ci  = (nu / si * 100) if si > 0 else 0
    period = vd["period"]

    ax.set_facecolor(PANEL_BG)
    ax.axhline(si, color=LGRAY, lw=0.7, ls="--", zorder=1)
    ax.axvline(ra, color=LGRAY, lw=0.7, ls="--", zorder=1)
    ax.axhline(nu, color=LGRAY, lw=0.6, ls=":",  zorder=1)
    ax.plot(0, nu, "o", ms=4.5, color="white",
            markeredgecolor=BLACK, markeredgewidth=1.0, zorder=6, clip_on=False)

    h_fit = np.linspace(lags[0] * 0.3, lags.max() * 1.04, 500)
    func  = MODELS[best]
    ax.plot(h_fit, func(h_fit, *params), color=BLACK, lw=1.6, zorder=3)
    ax.scatter(lags, func(lags, *params),
               marker="^", s=30, color=DGRAY, zorder=4, linewidths=0)
    ax.scatter(lags, gammas, s=34, color="white",
               edgecolors=BLACK, linewidths=1.1, zorder=5)

    rows = [
        ("Nugget", f"{nu:>{AW}.2f}"),
        ("Sill",   f"{si:>{AW}.2f}"),
        ("Range",  f"{ra:>{AW}.1f} m"),
        ("R\u00b2",    f"{r2:>{AW}.3f}"),
        ("CI",     f"{ci:>{AW}.1f} %"),
        ("Model",  f"{best:>{AW}}"),
    ]
    kw  = max(len(k) for k, _ in rows)
    txt = "\n".join(f"{k:<{kw}} = {v}" for k, v in rows)
    ax.text(0.975, 0.035, txt, transform=ax.transAxes,
            fontsize=FS_ANNOT, va="bottom", ha="right",
            fontfamily="monospace", fontstyle="normal", linespacing=1.5,
            bbox=dict(boxstyle="round,pad=0.42", fc="white",
                      ec=LGRAY, lw=0.7, alpha=0.96))

    ds = (period - 1) * 180
    de = min(period * 180, 1200)
    ax.set_title(f"Period {period}  (D{ds}\u2013D{de})",
                 fontsize=FS_TITLE, fontweight="bold", pad=4, color=BLACK)
    ax.set_xlabel("Distance (m)", fontsize=FS_LABEL, labelpad=3)
    if show_ylabel:
        ax.set_ylabel("Semivariance", fontsize=FS_LABEL, labelpad=3)
    ax.tick_params(labelsize=FS_TICK, pad=2)
    ax.set_xlim(left=0); ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(5, integer=True))


def _ax_pure_nugget(ax, period, show_ylabel):
    """Draw a 'pure nugget' placeholder panel."""
    ax.set_facecolor(PANEL_BG)
    ds = (period - 1) * 180
    de = min(period * 180, 1200)
    ax.set_title(f"Period {period}  (D{ds}\u2013D{de})",
                 fontsize=FS_TITLE, fontweight="bold", pad=4, color=BLACK)
    ax.text(0.5, 0.5, "Pure nugget effect\n(no spatial structure)",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=FS_ANNOT, color=DGRAY,
            bbox=dict(boxstyle="round,pad=0.5", fc="white", ec=LGRAY, lw=0.7))
    ax.set_xlabel("Distance (m)", fontsize=FS_LABEL, labelpad=3)
    if show_ylabel:
        ax.set_ylabel("Semivariance", fontsize=FS_LABEL, labelpad=3)
    ax.tick_params(labelsize=FS_TICK)


def _save(fig, basename):
    """Save figure as PNG and PDF."""
    for ext in ("png", "pdf"):
        path = os.path.join(OUTPUT_DIR, f"{basename}.{ext}")
        dpi  = DPI_FIGURE if ext == "png" else None
        fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {basename}.png / .pdf")


# =============================================================================
# 5. FIGURE A — SEMIVARIOGRAMS
# =============================================================================

def figure_semivariograms(vario_data, df, audpc_cols, lot_label):
    """
    2-row × 4-column grid.
    Panels 1–7: one semivariogram per period.
    Panel 8   : shared legend.
    """
    vd_map = {vd["period"]: vd for vd in vario_data}

    fig = plt.figure(figsize=(18, 10), facecolor="white", dpi=DPI_FIGURE)
    gs  = gridspec.GridSpec(2, 4, figure=fig,
                            hspace=0.38, wspace=0.28,
                            left=0.065, right=0.975,
                            top=0.920,  bottom=0.090)

    for idx in range(7):
        period = idx + 1
        r, c   = divmod(idx, 4)
        ax     = fig.add_subplot(gs[r, c])
        vd     = vd_map.get(period)
        col    = audpc_cols[idx] if idx < len(audpc_cols) else None
        std_v  = df[col].std() if col else 0

        if vd and not vd.get("pure_nugget", False) and std_v >= PURE_NUGGET_STD:
            _ax_variogram(ax, vd, show_ylabel=(c == 0))
        else:
            _ax_pure_nugget(ax, period, show_ylabel=(c == 0))

    # Legend in slot [1, 3]
    ax_leg = fig.add_subplot(gs[1, 3])
    ax_leg.set_visible(False)
    pos = ax_leg.get_position()
    leg = fig.legend(
        handles=LEGEND_HANDLES, fontsize=FS_LEG, loc="center",
        bbox_to_anchor=(pos.x0 + pos.width/2, pos.y0 + pos.height/2),
        bbox_transform=fig.transFigure,
        framealpha=0.95, edgecolor=LGRAY, fancybox=False,
        borderpad=0.9, handlelength=1.8, handletextpad=0.6,
        title="Legend", title_fontsize=FS_LEG,
    )
    leg.get_title().set_fontweight("bold")

    fig.text(0.003, 0.975, "A", fontsize=FS_PANEL, fontweight="bold",
             va="top", ha="left", color=BLACK)
    fig.suptitle(
        f"Empirical and fitted semivariograms of PRR severity (\u221aAUDPC) "
        f"across seven 180-day evaluation periods \u2014 Lot: {lot_label}",
        fontsize=12, fontweight="bold", y=0.998, color=BLACK,
    )

    _save(fig, f"Fig_A_{lot_label.replace(' ', '_')}")


# =============================================================================
# 6. FIGURE B — SPATIAL MAPS
# =============================================================================

def figure_maps(vario_data, df, audpc_cols, lot_label):
    """
    2-row × 4-column grid of scatter maps (one per period).
    Shared colour bar at the bottom; Morisita index annotated per panel.
    """
    vmin = 0
    vmax = float(np.ceil(df[audpc_cols].max().max()))
    norm = Normalize(vmin=vmin, vmax=vmax)
    pt   = max(1.5, min(6, 35_000 / len(df)))
    m_by = {vd["period"]: vd.get("morisita", np.nan) for vd in vario_data}

    fig2 = plt.figure(figsize=(18, 10), facecolor="white", dpi=DPI_FIGURE)
    gs2  = gridspec.GridSpec(2, 4, figure=fig2,
                             hspace=0.32, wspace=0.26,
                             left=0.065, right=0.975,
                             top=0.920,  bottom=0.055)

    for idx, col in enumerate(audpc_cols):
        r, c   = divmod(idx, 4)
        ax     = fig2.add_subplot(gs2[r, c])
        vals   = df[col].fillna(0).values
        period = idx + 1
        ds     = (period - 1) * 180
        de     = min(period * 180, 1200)

        ax.set_facecolor("#F8F8F8")
        ax.scatter(df["Easting"], df["Northing"],
                   c=vals, cmap=MAP_COLORMAP, norm=norm,
                   s=pt, alpha=0.85, linewidths=0, rasterized=True)
        ax.set_title(f"Period {period}  (D{ds}\u2013D{de})",
                     fontsize=FS_TITLE, fontweight="bold", pad=4, color=BLACK)
        ax.set_xlabel("Easting (m)",  fontsize=FS_LABEL, labelpad=3)
        if c == 0:
            ax.set_ylabel("Northing (m)", fontsize=FS_LABEL, labelpad=3)
        ax.tick_params(labelsize=FS_TICK - 1, pad=2)
        ax.set_aspect("equal", adjustable="datalim")
        ax.xaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))

        Im = m_by.get(period, np.nan)
        if not np.isnan(Im):
            ax.text(0.975, 0.965, f"I\u03b4 = {Im:.2f}",
                    transform=ax.transAxes, fontsize=FS_ANNOT,
                    fontstyle="normal", ha="right", va="top",
                    bbox=dict(boxstyle="round,pad=0.32",
                              fc="white", ec=LGRAY, lw=0.6, alpha=0.93))

    # Hide unused cell
    fig2.add_subplot(gs2[1, 3]).set_visible(False)

    # Shared colour bar
    cbar_ax = fig2.add_axes([0.065, 0.014, 0.660, 0.008])
    cb = fig2.colorbar(ScalarMappable(norm=norm, cmap=MAP_COLORMAP),
                       cax=cbar_ax, orientation="horizontal")
    cb.set_label("\u221aAUDPC", fontsize=FS_LABEL, labelpad=3)
    cb.ax.tick_params(labelsize=FS_TICK)
    cb.outline.set_linewidth(0.6)

    fig2.text(0.003, 0.975, "B", fontsize=FS_PANEL, fontweight="bold",
              va="top", ha="left", color=BLACK)
    fig2.suptitle(
        f"Spatial distribution of PRR severity (\u221aAUDPC) "
        f"across seven 180-day evaluation periods \u2014 Lot: {lot_label}",
        fontsize=12, fontweight="bold", y=0.998, color=BLACK,
    )

    _save(fig2, f"Fig_B_{lot_label.replace(' ', '_')}")


# =============================================================================
# 7. MODEL COMPARISON TABLE
# =============================================================================

PERIOD_ROW_COLORS = {
    "P1": "#F0F0F0", "P2": "white", "P3": "#F0F0F0",
    "P4": "white",   "P5": "#F0F0F0","P6": "white", "P7": "#F0F0F0",
}


def figure_model_table(all_results, lot_label):
    """
    Render the model comparison table as a figure (PNG + PDF).
    Best-fit rows are highlighted in dark grey.
    """
    df_res = pd.DataFrame(all_results)
    cols   = ["Period","Model","Nugget","Sill","Range (m)",
              "R²","RMSE","Cambardella (%)","Best"]
    data   = df_res[cols].values.tolist()

    fig, ax = plt.subplots(figsize=(16, 8), facecolor="white")
    ax.axis("off")
    tbl = ax.table(cellText=data, colLabels=cols, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.0, 1.9)

    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#AAAAAA"); cell.set_linewidth(0.5)
        if r == 0:
            cell.set_facecolor("black")
            cell.set_text_props(color="white", fontweight="bold", fontsize=10)
        else:
            row_d = data[r - 1]
            base  = PERIOD_ROW_COLORS.get(row_d[0], "white")
            if row_d[-1] == "*":
                cell.set_facecolor("#D0D0D0")
                cell.set_text_props(fontweight="bold")
            else:
                cell.set_facecolor(base)

    # Replace '*' with readable label
    for r in range(1, len(data) + 1):
        if data[r - 1][-1] == "*":
            tbl[r, 8].get_text().set_text("Best")
            tbl[r, 8].get_text().set_fontweight("bold")

    ax.set_title(
        f"Variogram model fit comparison \u2014 {lot_label}\n"
        "(Nugget, Sill, Range, R², RMSE, Cambardella index; "
        "shaded rows = best-fit model per period)",
        fontsize=11, fontweight="bold", pad=14, color="black",
    )
    plt.tight_layout()
    _save(fig, f"Table_{lot_label.replace(' ', '_')}")


# =============================================================================
# 8. MAIN PIPELINE
# =============================================================================

def run_lot(label, df):
    """Run the full analysis pipeline for one lot."""
    print(f"\n{'='*60}")
    print(f"  LOT: {label}")
    print(f"{'='*60}")

    df, audpc_cols = compute_audpc(df, PERIOD_BOUNDS)
    print(f"  AUDPC columns: {audpc_cols}")

    vario_data, all_results = analyse_lot(df, audpc_cols, label)

    figure_semivariograms(vario_data, df, audpc_cols, label)
    figure_maps(vario_data, df, audpc_cols, label)
    figure_model_table(all_results, label)

    return vario_data, all_results


if __name__ == "__main__":

    all_lot_results = []

    # ── Lot 1: Donmatias (original field data) ────────────────────────────────
    df_don = load_lot(DONMATIAS_XLSX, COORD_COLS_DONMATIAS, "Donmatias")
    vd_don, res_don = run_lot("Donmatias", df_don)
    all_lot_results.extend(res_don)

    # ── Lot 2: El Retiro ──────────────────────────────────────────────────────
    if EL_RETIRO_XLSX and os.path.exists(EL_RETIRO_XLSX):
        df_ret = load_lot(EL_RETIRO_XLSX, COORD_COLS_OTHERS, "El Retiro")
        vd_ret, res_ret = run_lot("El Retiro", df_ret)
        all_lot_results.extend(res_ret)
    else:
        print("\n[El Retiro] file not found — skipped.")

    # ── Lot 3: La Ceja ────────────────────────────────────────────────────────
    if LA_CEJA_XLSX and os.path.exists(LA_CEJA_XLSX):
        df_lac = load_lot(LA_CEJA_XLSX, COORD_COLS_OTHERS, "La Ceja")
        vd_lac, res_lac = run_lot("La Ceja", df_lac)
        all_lot_results.extend(res_lac)
    else:
        print("\n[La Ceja] file not found — skipped.")

    # ── Combined model comparison CSV ─────────────────────────────────────────
    csv_path = os.path.join(OUTPUT_DIR, "model_comparison_all_lots.csv")
    pd.DataFrame(all_lot_results).to_csv(csv_path, index=False)
    print(f"\nCombined table saved: {csv_path}")
    print("\nAll analyses complete.")

In [ ]:
"""
Geostatistical Kriging Analysis of PRR Severity
================================================
Full pipeline for three avocado production lots:
  1. Load data and compute √AUDPC per 180-day evaluation period
  2. Fit empirical variograms and select best theoretical model
     (Spherical, Exponential, Gaussian) by R²
  3. Run ordinary kriging using fitted variogram parameters
  4. Generate publication-quality kriging maps (viridis, white background,
     convex-hull masking, 3+3+1 panel layout per lot)

Lots
----
  - Donmatias  : Northing / Easting already in metres  (Taller_epi.xlsx)
  - El Retiro  : Latitude / Longitude  (Data_ElRetiro_1338.xlsx)
  - La Ceja    : Latitude / Longitude  (Data_LaCeja_1666.xlsx)

Outputs saved to OUTPUT_DIR
----------------------------
  Fig_Kriging_<Lot>.png / .pdf   — 7-panel kriging map per lot
  variogram_params_<Lot>.csv     — fitted variogram parameters per period

Dependencies
------------
    pip install pandas numpy scipy matplotlib openpyxl pykrige

Usage
-----
    python kriging_analysis.py
"""

import os
import warnings
import numpy  as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.optimize         import curve_fit
from scipy.spatial          import ConvexHull, Delaunay
from scipy.ndimage          import gaussian_filter
from pykrige.ok             import OrdinaryKriging
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
import matplotlib.patches   as mpatches
from matplotlib.colors import Normalize, LinearSegmentedColormap
warnings.filterwarnings("ignore")


# =============================================================================
# 0. USER CONFIGURATION
# =============================================================================

DONMATIAS_XLSX = "Taller_epi.xlsx"
EL_RETIRO_XLSX = "Data_ElRetiro_1338.xlsx"   # set None to skip
LA_CEJA_XLSX   = "Data_LaCeja_1666.xlsx"     # set None to skip

OUTPUT_DIR  = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Evaluation periods (days) — must match column names in the data
PERIOD_BOUNDS = [
    ("P1",   0,  180),
    ("P2", 180,  360),
    ("P3", 360,  540),
    ("P4", 540,  720),
    ("P5", 720,  900),
    ("P6", 900, 1080),
    ("P7",1080, 1200),
]

N_LAGS             = 12    # lag classes for empirical variogram
PERCENTILE_MAXDIST = 45    # % of pairwise distances used as max lag
PURE_NUGGET_STD    = 0.05  # std below which a period is flagged pure-nugget
GRID_STEPS         = 80    # kriging interpolation grid resolution
SMOOTH_SIGMA       = 0.6   # Gaussian smoothing (sigma in grid cells) for display

DPI = 400


# =============================================================================
# 1. DATA LOADING
# =============================================================================

def lat_lon_to_metres(df, lat_col="Latitude", lon_col="Longitude"):
    """Convert lat/lon to local metric coordinates centred on the lot."""
    lat0 = df[lat_col].mean()
    lon0 = df[lon_col].mean()
    cos_lat = np.cos(np.radians(lat0))
    df = df.copy()
    df["Northing"] = (df[lat_col] - lat0) * 111_320
    df["Easting"]  = (df[lon_col] - lon0) * 111_320 * cos_lat
    return df


def load_lot(xlsx_path, label, has_utm=True):
    """
    Load a lot from Excel.
    has_utm=True  → Northing/Easting columns already in metres.
    has_utm=False → Latitude/Longitude converted to metres.
    """
    df = pd.read_excel(xlsx_path)
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    if not has_utm:
        df = lat_lon_to_metres(df)
    print(f"  [{label}] {len(df)} plants loaded from {xlsx_path}")
    return df


# =============================================================================
# 2. AUDPC COMPUTATION
# =============================================================================

def compute_audpc(df, period_bounds=PERIOD_BOUNDS):
    """
    Add √AUDPC columns for each 180-day evaluation period.
    Returns augmented DataFrame and list of √AUDPC column names.
    """
    disease_cols = sorted(
        [c for c in df.columns if c.startswith("D") and c[1:].isdigit()],
        key=lambda x: int(x[1:])
    )
    for pname, lo, hi in period_bounds:
        cols = [c for c in disease_cols if lo <= int(c[1:]) <= hi]
        if len(cols) < 2:
            continue
        times = [int(c[1:]) for c in cols]
        vals  = df[cols].values
        df[f"AUDPC_{pname}"]      = np.trapezoid(vals, times, axis=1)
        df[f"AUDPC_{pname}_sqrt"] = np.sqrt(np.maximum(df[f"AUDPC_{pname}"], 0))

    audpc_cols = [c for c in df.columns if "AUDPC_P" in c and "sqrt" in c]
    return df, audpc_cols


# =============================================================================
# 3. EMPIRICAL VARIOGRAM
# =============================================================================

def empirical_variogram(coords, values,
                         n_lags=N_LAGS,
                         pct_maxdist=PERCENTILE_MAXDIST):
    """
    Method-of-moments empirical variogram (Matheron 1963).

    Parameters
    ----------
    coords  : (n, 2) array  — spatial coordinates in metres
    values  : (n,)   array  — observed values
    n_lags  : int           — number of lag classes
    pct_maxdist : float     — upper percentile of distances used as max lag

    Returns
    -------
    lags, gammas, counts : 1-D arrays
    """
    dists    = cdist(coords, coords)
    vals     = np.asarray(values, dtype=float)
    max_dist = np.percentile(dists[dists > 0], pct_maxdist)
    lag_size = max_dist / n_lags
    lags, gammas, counts = [], [], []
    for i in range(n_lags):
        lo, hi = i * lag_size, (i + 1) * lag_size
        mask   = (dists > lo) & (dists <= hi)
        pi, pj = np.where(mask)
        if len(pi) > 10:
            lags.append((lo + hi) / 2)
            gammas.append(((vals[pi] - vals[pj])**2).mean() / 2)
            counts.append(len(pi))
    return np.array(lags), np.array(gammas), np.array(counts)


# =============================================================================
# 4. THEORETICAL VARIOGRAM MODELS
# =============================================================================

def model_spherical(h, nugget, sill, rang):
    """Spherical variogram model."""
    return np.where(
        h <= rang,
        nugget + (sill - nugget) * (1.5*(h/rang) - 0.5*(h/rang)**3),
        sill
    )

def model_exponential(h, nugget, sill, rang):
    """Exponential variogram model."""
    return nugget + (sill - nugget) * (1 - np.exp(-h / rang))

def model_gaussian(h, nugget, sill, rang):
    """Gaussian variogram model."""
    return nugget + (sill - nugget) * (1 - np.exp(-(h / rang)**2))

MODELS = {
    "Spherical":   model_spherical,
    "Exponential": model_exponential,
    "Gaussian":    model_gaussian,
}

# PyKrige model name mapping
PYKRIGE_MODEL = {
    "Spherical":   "spherical",
    "Exponential": "exponential",
    "Gaussian":    "gaussian",
}


def fit_variogram_models(lags, gammas):
    """
    Fit all three theoretical models by non-linear least squares.
    Select best model by R².

    Returns
    -------
    best_name   : str
    best_params : (nugget, sill, range)
    best_r2     : float
    all_results : dict  {name: {'params', 'r2', 'rmse'} | None}
    """
    s0 = gammas.max(); r0 = lags.max() * 0.5; n0 = gammas.min() * 0.1
    all_results = {}
    for name, func in MODELS.items():
        try:
            popt, _ = curve_fit(
                func, lags, gammas,
                p0=[n0, s0, r0],
                bounds=([0, 0, 0], [s0*2, s0*3, lags.max()*2]),
                maxfev=5000,
            )
            pred   = func(lags, *popt)
            ss_res = np.sum((gammas - pred)**2)
            ss_tot = np.sum((gammas - gammas.mean())**2)
            r2     = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
            rmse   = np.sqrt(ss_res / len(lags))
            all_results[name] = dict(params=popt, r2=r2, rmse=rmse)
        except Exception:
            all_results[name] = None

    valid = {k: v for k, v in all_results.items() if v is not None}
    if not valid:
        return None, None, None, all_results
    best_name = max(valid, key=lambda k: valid[k]["r2"])
    p         = valid[best_name]
    return best_name, p["params"], p["r2"], all_results


def run_variogram_analysis(df, audpc_cols, label):
    """
    Compute empirical variograms and fit theoretical models for all periods.

    Returns
    -------
    vario_results : list of per-period dicts
    params_rows   : list of dicts for CSV export
    """
    coords       = df[["Northing", "Easting"]].values
    vario_results = []
    params_rows   = []

    for idx, col in enumerate(audpc_cols):
        period  = idx + 1
        vals    = df[col].fillna(0).values.astype(float)
        std_val = vals.std()
        print(f"  [{label}] P{period}  std={std_val:.3f}", end="")

        if std_val < PURE_NUGGET_STD:
            print(" → pure nugget (no spatial structure)")
            vario_results.append(dict(period=period, pure_nugget=True,
                                      best_name=None, params=None, r2=None))
            params_rows.append(dict(Lot=label, Period=f"P{period}",
                                    Model="Pure nugget", Nugget=np.nan,
                                    Sill=np.nan, Range_m=np.nan,
                                    R2=np.nan, RMSE=np.nan))
            continue

        lags, gammas, counts = empirical_variogram(coords, vals)
        if len(lags) < 3:
            print(" → too few lag classes")
            continue

        best_name, params, r2, all_res = fit_variogram_models(lags, gammas)
        if best_name is None:
            print(" → no model converged")
            continue

        nu, si, ra = params
        ci = (nu / si * 100) if si > 0 else 0
        print(f" → best={best_name}  R²={r2:.3f}  "
              f"nugget={nu:.2f}  sill={si:.2f}  range={ra:.1f} m  CI={ci:.1f}%")

        vario_results.append(dict(
            period=period, pure_nugget=False,
            lags=lags, gammas=gammas, counts=counts,
            best_name=best_name, params=params,
            r2=r2, all_results=all_res,
        ))

        for mname, mres in all_res.items():
            if mres:
                nu_m, si_m, ra_m = mres["params"]
                params_rows.append(dict(
                    Lot=label, Period=f"P{period}", Model=mname,
                    Nugget=round(nu_m, 2), Sill=round(si_m, 2),
                    Range_m=round(ra_m, 1),
                    R2=round(mres["r2"], 4), RMSE=round(mres["rmse"], 4),
                    Cambardella_pct=round((nu_m/si_m*100) if si_m>0 else 0, 1),
                    Best="*" if mname == best_name else "",
                ))

    return vario_results, params_rows


# =============================================================================
# 5. ORDINARY KRIGING INTERPOLATION
# =============================================================================

def run_kriging(df, vario_results, audpc_cols,
                grid_steps=GRID_STEPS, label=""):
    """
    Perform ordinary kriging for each period using fitted variogram parameters.
    Falls back to inverse-distance weighting (IDW) if kriging fails.

    Returns
    -------
    list of per-period dicts:
        period, z_grid, xi, yi, x_obs, y_obs, z_obs
    """
    x_raw = df["Easting"].values.astype(float)
    y_raw = df["Northing"].values.astype(float)
    x_min, x_max = x_raw.min(), x_raw.max()
    y_min, y_max = y_raw.min(), y_raw.max()
    span = max(x_max - x_min, y_max - y_min)

    # Normalise to [0,1] for numerical stability
    x_n  = (x_raw - x_min) / span
    y_n  = (y_raw - y_min) / span
    xi_n = np.linspace(0, (x_max - x_min) / span, grid_steps)
    yi_n = np.linspace(0, (y_max - y_min) / span, grid_steps)

    vd_map  = {v["period"]: v for v in vario_results}
    results = []

    for idx, col in enumerate(audpc_cols):
        period = idx + 1
        z      = df[col].fillna(0).values.astype(float)
        vd     = vd_map.get(period, {})
        z_grid = None

        # ── Ordinary kriging ──────────────────────────────────────────────────
        if (not vd.get("pure_nugget", False)
                and vd.get("best_name") is not None):
            nu, si, ra = vd["params"]
            pykrige_m  = PYKRIGE_MODEL[vd["best_name"]]
            # PyKrige parameter order: [psill, range, nugget]
            psill = max(si - nu, 0.01)
            rang  = max(ra / span, 1e-4)
            nug   = max(nu, 0.0)
            try:
                OK = OrdinaryKriging(
                    x_n, y_n, z,
                    variogram_model=pykrige_m,
                    variogram_parameters=[psill, rang, nug],
                    verbose=False, enable_plotting=False,
                )
                z_grid_raw, _ = OK.execute("grid", xi_n, yi_n)
                z_grid = np.clip(np.array(z_grid_raw), 0, None)
                # Cap artefacts at 1.5× observed max
                cap    = z.max() * 1.5
                z_grid = np.clip(z_grid, 0, cap)
            except Exception as e:
                print(f"  [{label}] P{period}: kriging failed ({e}) → IDW")

        # ── IDW fallback ──────────────────────────────────────────────────────
        if z_grid is None:
            XX, YY = np.meshgrid(xi_n, yi_n)
            z_grid = np.zeros_like(XX, dtype=float)
            for i in range(grid_steps):
                for j in range(grid_steps):
                    d2 = (x_n - XX[i, j])**2 + (y_n - YY[i, j])**2
                    d2 = np.maximum(d2, 1e-10)
                    z_grid[i, j] = np.sum(z / d2) / np.sum(1.0 / d2)
            z_grid = np.clip(z_grid, 0, None)

        results.append(dict(
            period=period, col=col,
            z_grid=z_grid,
            xi=xi_n * (x_max - x_min) + x_min,
            yi=yi_n * (y_max - y_min) + y_min,
            x_obs=x_raw, y_obs=y_raw, z_obs=z,
        ))

    return results


# =============================================================================
# 6. CONVEX HULL MASK
# =============================================================================

def hull_mask(xi, yi, x_obs, y_obs):
    """
    Boolean mask (len(yi) × len(xi)) — True inside the convex hull of
    observation points.  Returns all-True on failure.
    """
    pts = np.column_stack([x_obs, y_obs])
    try:
        hull = ConvexHull(pts)
        tri  = Delaunay(pts[hull.vertices])
        XI, YI = np.meshgrid(xi, yi)
        inside = tri.find_simplex(
            np.column_stack([XI.ravel(), YI.ravel()])
        ) >= 0
        return inside.reshape(XI.shape)
    except Exception:
        return np.ones((len(yi), len(xi)), dtype=bool)


# =============================================================================
# 7. FIGURE — KRIGING MAPS
# =============================================================================

CMAP_PRR = plt.cm.viridis

plt.rcParams.update({
    "font.family":      "sans-serif",
    "font.sans-serif":  ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":        9,
    "axes.facecolor":   "white",
    "figure.facecolor": "white",
    "axes.grid":        True,
    "grid.color":       "#DDDDDD",
    "grid.linewidth":   0.5,
    "grid.linestyle":   "-",
    "axes.linewidth":   0.5,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "xtick.major.width":0.5, "ytick.major.width":0.5,
    "xtick.direction":  "out", "ytick.direction": "out",
    "pdf.fonttype":     42,   "ps.fonttype":      42,
})

BLACK      = "#000000"
PANEL_LBL  = list("ABCDEFG")
# Panel positions in a 3×3 grid (row, col) for 7 periods
POSITIONS  = [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2),(2,1)]


def _make_discrete_legend(ax, vmin, vmax, cmap, n_bins=5):
    """Discrete colour legend matching the reference style."""
    bounds  = np.linspace(vmin, vmax, n_bins + 1)
    labels  = [f"{bounds[i]:.2f} – {bounds[i+1]:.2f}"
               for i in range(n_bins)]
    colors  = [cmap(i / (n_bins - 1)) for i in range(n_bins)]
    patches = [mpatches.Patch(facecolor=colors[i], edgecolor="none",
                               label=labels[i])
               for i in range(n_bins - 1, -1, -1)]
    leg = ax.legend(
        handles=patches,
        title="Severity (√AUDPC)",
        loc="lower right",
        fontsize=6.5,
        title_fontsize=7,
        framealpha=0.92,
        edgecolor="#AAAAAA",
        fancybox=False,
        borderpad=0.6,
        handlelength=1.2,
        handletextpad=0.5,
    )
    leg.get_title().set_fontweight("bold")
    return leg


def figure_kriging(lot_label, krig_results, vmin=0.0, vmax=None):
    """
    7-panel kriging figure (3+3+1 layout) for one lot.
    Saves PNG (400 dpi) and PDF.
    """
    if vmax is None:
        vmax = float(np.percentile(
            [k["z_grid"].max() for k in krig_results], 95))
    norm = Normalize(vmin=vmin, vmax=vmax)

    fig = plt.figure(figsize=(11, 14), facecolor="white", dpi=DPI)
    gs  = gridspec.GridSpec(3, 3, figure=fig,
                             hspace=0.18, wspace=0.12,
                             left=0.10, right=0.95,
                             top=0.93,  bottom=0.06)

    for pidx, kdat in enumerate(krig_results):
        r, c    = POSITIONS[pidx]
        ax      = fig.add_subplot(gs[r, c])
        period  = kdat["period"]
        xi, yi  = kdat["xi"], kdat["yi"]
        zg      = gaussian_filter(kdat["z_grid"], sigma=SMOOTH_SIGMA)
        zg      = np.clip(zg, 0, None)
        xo, yo, zo = kdat["x_obs"], kdat["y_obs"], kdat["z_obs"]

        # Apply convex hull mask (outside → NaN → white)
        mask = hull_mask(xi, yi, xo, yo)
        zg_m = np.where(mask, zg, np.nan)

        # ── Kriging surface ───────────────────────────────────────────────────
        ax.imshow(
            zg_m, origin="lower",
            extent=[xi.min(), xi.max(), yi.min(), yi.max()],
            cmap=CMAP_PRR, norm=norm, aspect="auto",
            interpolation="bilinear", zorder=2,
        )

        # ── Grid behind image ────────────────────────────────────────────────
        ax.set_axisbelow(True)
        ax.grid(True, color="#DDDDDD", lw=0.5, zorder=1)

        # ── Observation points ───────────────────────────────────────────────
        ax.scatter(xo, yo, c=zo, cmap=CMAP_PRR, norm=norm,
                   s=1.4, linewidths=0, zorder=3, alpha=0.45)

        # ── Panel letter ─────────────────────────────────────────────────────
        ax.text(0.03, 0.97, PANEL_LBL[pidx],
                transform=ax.transAxes,
                fontsize=10, fontweight="bold", color=BLACK,
                ha="left", va="top", zorder=5)

        # ── Period label ──────────────────────────────────────────────────────
        ds = (period - 1) * 180
        de = min(period * 180, 1200)
        ax.text(0.97, 0.97, f"D{ds}–D{de}",
                transform=ax.transAxes, fontsize=6.5, color=BLACK,
                ha="right", va="top",
                bbox=dict(boxstyle="round,pad=0.2",
                          fc="white", ec="none", alpha=0.75))

        # ── Ticks and labels ──────────────────────────────────────────────────
        ax.tick_params(labelsize=6.5, colors=BLACK, pad=2)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(5, integer=True))

        # X-label only bottom panels
        if r == 2 or (r == 1 and c in [0, 1, 2]):
            ax.set_xlabel("Spatial dimension in X",
                          fontsize=7.5, labelpad=3, color=BLACK)
        # Y-label only left column
        if c == 0:
            ax.set_ylabel("Spatial dimension in Y",
                          fontsize=7.5, labelpad=3, color=BLACK)

        ax.set_facecolor("white")
        for sp in ax.spines.values():
            sp.set_color("#BBBBBB"); sp.set_linewidth(0.5)

        # Discrete legend only on last panel (G)
        if pidx == 6:
            _make_discrete_legend(ax, vmin, vmax, CMAP_PRR, n_bins=5)

    # Hide unused cells in last row
    for c_hide in [0, 2]:
        ax_h = fig.add_subplot(gs[2, c_hide])
        ax_h.set_visible(False)

    fig.suptitle(
        f"Ordinary kriging interpolation of PRR severity (√AUDPC)\n"
        f"Lot: {lot_label}",
        fontsize=11, fontweight="bold", y=0.97,
        color=BLACK, linespacing=1.35,
    )

    tag  = lot_label.replace(" ", "_")
    _save(fig, f"Fig_Kriging_{tag}")


def _save(fig, basename):
    for ext in ("png", "pdf"):
        path = os.path.join(OUTPUT_DIR, f"{basename}.{ext}")
        fig.savefig(path, dpi=DPI if ext == "png" else None,
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {basename}.png / .pdf")


# =============================================================================
# 8. MAIN PIPELINE
# =============================================================================

if __name__ == "__main__":

    lots_config = [
        (DONMATIAS_XLSX, "Donmatias", True),
        (EL_RETIRO_XLSX, "El Retiro", False),
        (LA_CEJA_XLSX,   "La Ceja",   False),
    ]

    for xlsx, label, has_utm in lots_config:

        if xlsx is None or not os.path.exists(xlsx):
            print(f"\n[{label}] file not found — skipped.")
            continue

        print(f"\n{'='*60}")
        print(f"  LOT: {label}")
        print(f"{'='*60}")

        # ── Step 1: load and compute AUDPC ─────────────────────────────────────
        df = load_lot(xlsx, label, has_utm=has_utm)
        df, audpc_cols = compute_audpc(df)
        print(f"  AUDPC columns: {audpc_cols}")

        # ── Step 2: variogram analysis ─────────────────────────────────────────
        print("\n  Variogram fitting:")
        vario_results, params_rows = run_variogram_analysis(
            df, audpc_cols, label)

        # Save variogram parameters
        csv_path = os.path.join(OUTPUT_DIR,
                                f"variogram_params_{label.replace(' ','_')}.csv")
        pd.DataFrame(params_rows).to_csv(csv_path, index=False)
        print(f"  Parameters saved: {csv_path}")

        # ── Step 3: kriging interpolation ─────────────────────────────────────
        print("\n  Kriging interpolation:")
        krig_results = run_kriging(df, vario_results, audpc_cols, label=label)

        # ── Step 4: figure ─────────────────────────────────────────────────────
        print("\n  Generating figure:")
        figure_kriging(label, krig_results)

    print("\nAll analyses complete.")

In [ ]:
"""
Getis-Ord Gi* Hotspot Analysis of PRR Disease
===============================================
Point-pattern spatial analysis for three avocado production lots using the
Getis-Ord Gi* statistic to identify statistically significant hotspots,
coldspots and non-significant areas of disease activity.

The distance threshold for spatial weights is set to the median range of
spatial dependence estimated from fitted variogram models (biologically
informed bandwidth).

Lots
----
  - Donmatias  : Northing/Easting already in metres  (Taller_epi.xlsx)
  - El Retiro  : Latitude/Longitude                  (Data_ElRetiro_1338.xlsx)
  - La Ceja    : Latitude/Longitude                  (Data_LaCeja_1666.xlsx)

Outputs saved to OUTPUT_DIR
----------------------------
  Fig_GiStar_<Lot>.png / .pdf   — 2×4 panel Gi* maps for 8 time snapshots
  Table_GiStar_<Lot>.png / .pdf — classification counts per time point
  gistar_results_<Lot>.csv      — full z-score and classification per plant

Dependencies
------------
    pip install pandas numpy scipy matplotlib openpyxl

Usage
-----
    python gistar_hotspot_analysis.py

Notes
-----
The Getis-Ord Gi* statistic (Getis & Ord 1992; Ord & Getis 1995) for each
location i is:

           Σj wij(d) xj  −  x̄ Σj wij(d)
    Gi* = ─────────────────────────────────────────────────────
           s √[(n Σj wij²(d) − (Σj wij(d))²) / (n − 1)]

where wij(d) = 1 if dist(i,j) ≤ d, 0 otherwise (including self, i.e. wii = 1),
x̄ and s are the global mean and standard deviation, and n is the number of
observations.  The statistic is approximately standard normal under the null
hypothesis of spatial randomness.

Classification thresholds (two-tailed):
    z > +2.58  →  Hot Spot  (99% confidence, p < 0.01)
    z > +1.96  →  Hot Spot  (95% confidence, p < 0.05)
    |z| ≤ 1.96 →  Not Significant
    z < −1.96  →  Cold Spot (95% confidence, p < 0.05)
    z < −2.58  →  Cold Spot (99% confidence, p < 0.01)
"""

import os
import warnings
import numpy  as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.stats             import norm as sp_norm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
from matplotlib.lines import Line2D
warnings.filterwarnings("ignore")


# =============================================================================
# 0. USER CONFIGURATION
# =============================================================================

DONMATIAS_XLSX = "Taller_epi.xlsx"
EL_RETIRO_XLSX = "Data_ElRetiro_1338.xlsx"   #Data not included due to confidentiality
LA_CEJA_XLSX   = "Data_LaCeja_1666.xlsx"     #Data not included due to confidentiality

OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Time snapshots to display in the figure (days)
# These must correspond to column names D<t> in the data
TIME_POINTS_SHOW = [0, 180, 360, 540, 720, 900, 1080, 1200]

# Distance threshold per lot (metres) — median variogram range.
# These were estimated from the fitted semivariogram models.
# Adjust if you refit variograms on your own data.
DISTANCE_THRESHOLDS = {
    "Donmatias": 46.5,
    "El Retiro":  40.6,
    "La Ceja":    37.8,
}

# Alternatively, set a single threshold for all lots (overrides the dict above)
# GLOBAL_THRESHOLD = 45.0   # uncomment to use

# Statistical significance thresholds (z-scores, two-tailed)
Z_99 = 2.576   # p < 0.01
Z_95 = 1.960   # p < 0.05

DPI = 400   # figure resolution


# =============================================================================
# 1. DATA LOADING
# =============================================================================

def lat_lon_to_metres(df, lat_col="Latitude", lon_col="Longitude"):
    """Convert lat/lon to local metric coordinates centred on lot centroid."""
    lat0    = df[lat_col].mean()
    lon0    = df[lon_col].mean()
    cos_lat = np.cos(np.radians(lat0))
    df      = df.copy()
    df["Northing"] = (df[lat_col] - lat0) * 111_320
    df["Easting"]  = (df[lon_col] - lon0) * 111_320 * cos_lat
    return df


def load_lot(xlsx_path, label, has_utm=True):
    """
    Load a lot from Excel.
    has_utm=True  → Northing/Easting already in metres.
    has_utm=False → Latitude/Longitude; converted to local metres.
    """
    df = pd.read_excel(xlsx_path)
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    if not has_utm:
        df = lat_lon_to_metres(df)
    # Centre coordinates for numerical stability
    df["Northing"] -= df["Northing"].mean()
    df["Easting"]  -= df["Easting"].mean()
    print(f"  [{label}] {len(df)} plants loaded from {xlsx_path}")
    return df


def get_disease_cols(df, time_points):
    """Return disease column names present in df for the given time points."""
    return [f"D{t}" for t in time_points if f"D{t}" in df.columns]


# =============================================================================
# 2. GETIS-ORD Gi* STATISTIC
# =============================================================================

def getis_ord_gi_star(coords, values, distance_threshold):
    """
    Compute the Getis-Ord Gi* statistic for each observation.

    The spatial weight matrix is binary: wij = 1 if dist(i,j) ≤ d (including
    self, wii = 1), 0 otherwise.  This is the standard formulation for Gi*
    (Ord & Getis 1995).

    Parameters
    ----------
    coords             : (n, 2) float array — spatial coordinates [Easting, Northing]
    values             : (n,)   float array — observed values (e.g. severity 0–5)
    distance_threshold : float  — bandwidth d in the same units as coords

    Returns
    -------
    z_scores : (n,) float array — standardised Gi* z-scores
    p_values : (n,) float array — two-tailed approximate p-values
    """
    n    = len(values)
    x    = values.astype(float)
    xbar = x.mean()
    s    = x.std(ddof=0)

    # Binary weight matrix (include self by setting diagonal = 1)
    dists = cdist(coords, coords)
    W     = (dists <= distance_threshold).astype(float)

    wi_sum  = W.sum(axis=1)      # Σj wij  (scalar per row)
    wi_sum2 = (W**2).sum(axis=1) # Σj wij² (scalar per row)
    wx      = W @ x              # Σj wij xj (local weighted sum)

    numerator   = wx - xbar * wi_sum
    denominator = s * np.sqrt(
        np.maximum((n * wi_sum2 - wi_sum**2) / (n - 1), 1e-10)
    )
    z_scores = np.where(denominator > 0, numerator / denominator, 0.0)
    p_values = 2.0 * (1.0 - sp_norm.cdf(np.abs(z_scores)))

    return z_scores, p_values


def classify_gi_star(z_scores, p_values, z_99=Z_99, z_95=Z_95):
    """
    Classify each observation into one of five categories based on Gi* z-score.

    Returns
    -------
    categories : (n,) object array of str
    """
    cats = np.full(len(z_scores), "Not Significant", dtype=object)
    cats[(z_scores >  z_95) & (p_values < 0.05)] = "Hot Spot (95%)"
    cats[(z_scores >  z_99) & (p_values < 0.01)] = "Hot Spot (99%)"
    cats[(z_scores < -z_95) & (p_values < 0.05)] = "Cold Spot (95%)"
    cats[(z_scores < -z_99) & (p_values < 0.01)] = "Cold Spot (99%)"
    return cats


def run_gi_star_all_periods(df, time_points, distance_threshold, label):
    """
    Compute Gi* for every requested time point.

    Returns
    -------
    results : dict  {time_point: {'z': z_scores, 'p': p_values, 'cat': categories}}
    summary : list of dicts (for CSV export)
    """
    coords  = df[["Easting", "Northing"]].values
    results = {}
    summary = []

    for t in time_points:
        col = f"D{t}"
        if col not in df.columns:
            print(f"  [{label}] D{t}: column not found — skipped")
            continue
        values = df[col].fillna(0).values.astype(float)

        z, p = getis_ord_gi_star(coords, values, distance_threshold)
        cats  = classify_gi_star(z, p)

        # Per-category counts
        unique, counts = np.unique(cats, return_counts=True)
        count_dict = dict(zip(unique, counts))

        n_hs99 = count_dict.get("Hot Spot (99%)",  0)
        n_hs95 = count_dict.get("Hot Spot (95%)",  0)
        n_ns   = count_dict.get("Not Significant", 0)
        n_cs95 = count_dict.get("Cold Spot (95%)", 0)
        n_cs99 = count_dict.get("Cold Spot (99%)", 0)

        print(f"  [{label}] D{t:>4}:  "
              f"HS99={n_hs99:4d}  HS95={n_hs95:3d}  "
              f"NS={n_ns:4d}  CS95={n_cs95:3d}  CS99={n_cs99:4d}")

        results[t] = {"z": z, "p": p, "cat": cats,
                      "values": values, "d_thresh": distance_threshold}
        summary.append(dict(
            Time_days=t, Distance_threshold_m=distance_threshold,
            n_HotSpot_99=n_hs99, n_HotSpot_95=n_hs95,
            n_NotSignificant=n_ns,
            n_ColdSpot_95=n_cs95, n_ColdSpot_99=n_cs99,
            pct_HotSpot=round((n_hs99 + n_hs95) / len(z) * 100, 1),
            pct_ColdSpot=round((n_cs99 + n_cs95) / len(z) * 100, 1),
            mean_severity=round(values.mean(), 4),
        ))

    return results, summary


# =============================================================================
# 3. EXPORT FULL RESULTS CSV
# =============================================================================

def export_results_csv(df, gi_results, label):
    """
    Export a CSV with plant-level z-scores and classifications for all periods.
    """
    out = df[["Easting", "Northing"]].copy()
    # Add original disease columns if present
    dcols = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]
    out = pd.concat([out, df[dcols]], axis=1)

    for t, res in gi_results.items():
        out[f"GiStar_z_D{t}"]   = res["z"].round(4)
        out[f"GiStar_p_D{t}"]   = res["p"].round(6)
        out[f"GiStar_cat_D{t}"] = res["cat"]

    path = os.path.join(
        OUTPUT_DIR, f"gistar_results_{label.replace(' ', '_')}.csv")
    out.to_csv(path, index=False)
    print(f"  Results saved: {path}")


# =============================================================================
# 4. GLOBAL PLOT STYLE
# =============================================================================

plt.rcParams.update({
    "font.family":       "sans-serif",
    "font.sans-serif":   ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size":         9,
    "axes.linewidth":    0.6,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "xtick.major.size":  2.5, "ytick.major.size":  2.5,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.direction":   "out", "ytick.direction":  "out",
    "pdf.fonttype":      42,   "ps.fonttype":       42,
})

BLACK = "#000000"
LGRAY = "#CCCCCC"

# Visual style per classification category
CATEGORY_STYLE = {
    "Hot Spot (99%)":  dict(color="#B22222", s=6,  alpha=0.90, zorder=5),
    "Hot Spot (95%)":  dict(color="#FF6B6B", s=5,  alpha=0.80, zorder=4),
    "Not Significant": dict(color="#AAAAAA", s=3,  alpha=0.55, zorder=2),
    "Cold Spot (95%)": dict(color="#6BAED6", s=5,  alpha=0.80, zorder=4),
    "Cold Spot (99%)": dict(color="#08519C", s=6,  alpha=0.90, zorder=5),
}

# Draw order: background to foreground
DRAW_ORDER = [
    "Not Significant",
    "Cold Spot (95%)", "Cold Spot (99%)",
    "Hot Spot (95%)",  "Hot Spot (99%)",
]

LEGEND_HANDLES = [
    Line2D([0],[0], marker="o", color="w", markersize=6,
           markerfacecolor=CATEGORY_STYLE[c]["color"], label=c)
    for c in ["Hot Spot (99%)", "Hot Spot (95%)",
              "Not Significant",
              "Cold Spot (95%)", "Cold Spot (99%)"]
]


def _save(fig, basename):
    for ext in ("png", "pdf"):
        path = os.path.join(OUTPUT_DIR, f"{basename}.{ext}")
        fig.savefig(path, dpi=DPI if ext == "png" else None,
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Figure saved: {basename}.png / .pdf")


# =============================================================================
# 5. FIGURE — Gi* MAPS  (2×4 panel layout)
# =============================================================================

def figure_gistar_maps(df, gi_results, lot_label, distance_threshold):
    """
    2-row × 4-column figure showing Gi* classification maps for 8 time
    snapshots.  Points are coloured by classification category.
    The last panel includes the shared legend.
    """
    x = df["Easting"].values
    y = df["Northing"].values

    time_pts = sorted(gi_results.keys())
    n_panels = len(time_pts)
    ncols    = 4
    nrows    = int(np.ceil(n_panels / ncols))

    fig = plt.figure(figsize=(18, 5 * nrows), facecolor="white", dpi=DPI)
    gs  = gridspec.GridSpec(nrows, ncols, figure=fig,
                             hspace=0.26, wspace=0.20,
                             left=0.06, right=0.97,
                             top=0.90, bottom=0.06)

    for pidx, t in enumerate(time_pts):
        row, col = divmod(pidx, ncols)
        ax       = fig.add_subplot(gs[row, col])
        res      = gi_results[t]
        cats     = res["cat"]

        # Plot each category in draw order
        for cat_name in DRAW_ORDER:
            mask = cats == cat_name
            if mask.sum() == 0:
                continue
            st = CATEGORY_STYLE[cat_name]
            ax.scatter(x[mask], y[mask],
                       c=st["color"], s=st["s"],
                       alpha=st["alpha"], linewidths=0, zorder=st["zorder"])

        # Panel title
        ax.set_title(f"{t} days", fontsize=10, fontweight="bold",
                     pad=4, color=BLACK)

        # Axis labels: bottom row and left column only
        if row == nrows - 1:
            ax.set_xlabel("Easting (m)", fontsize=8, labelpad=2, color=BLACK)
        if col == 0:
            ax.set_ylabel("Northing (m)", fontsize=8, labelpad=2, color=BLACK)

        ax.tick_params(labelsize=7.5, colors=BLACK, pad=2)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
        ax.set_aspect("equal", adjustable="datalim")
        ax.set_facecolor("white")
        for sp in ax.spines.values():
            sp.set_color(LGRAY); sp.set_linewidth(0.5)

        # Legend on last panel
        if pidx == n_panels - 1:
            ax.legend(
                handles=LEGEND_HANDLES,
                fontsize=7.5, loc="lower right",
                title="Gi* classification",
                title_fontsize=8,
                framealpha=0.92, edgecolor=LGRAY, fancybox=False,
                borderpad=0.6, handletextpad=0.5,
            )

    fig.suptitle(
        f"Getis-Ord Gi* hotspot analysis of PRR severity — Lot: {lot_label}\n"
        f"(distance threshold = {distance_threshold} m, "
        "biologically informed from variogram range)",
        fontsize=11, fontweight="bold", y=0.995,
        color=BLACK, linespacing=1.35,
    )
    _save(fig, f"Fig_GiStar_{lot_label.replace(' ', '_')}")


# =============================================================================
# 6. FIGURE — SUMMARY TABLE
# =============================================================================

TABLE_ROW_COLORS_T = {
    0:    "white",
    180:  "#FFF3B0",
    360:  "#FFF3B0",
    540:  "#FFE0B2",
    720:  "#FFE0B2",
    900:  "#FFCCBC",
    1080: "#FFCCBC",
    1200: "#FFCDD2",
}


def figure_summary_table(summary_rows, lot_label, distance_threshold):
    """
    Render a summary table of Gi* classification counts per time point.
    """
    df_s = pd.DataFrame(summary_rows)
    cols = ["Time_days", "Distance_threshold_m",
            "n_HotSpot_99", "n_HotSpot_95", "n_NotSignificant",
            "n_ColdSpot_95", "n_ColdSpot_99",
            "pct_HotSpot", "pct_ColdSpot", "mean_severity"]
    col_labels = ["Time (days)", "Threshold (m)",
                  "Hot (99%)", "Hot (95%)", "Not Sig.",
                  "Cold (95%)", "Cold (99%)",
                  "% Hot", "% Cold", "Mean sev."]
    data = df_s[cols].values.tolist()

    fig, ax = plt.subplots(figsize=(16, 6), facecolor="white")
    ax.axis("off")
    tbl = ax.table(cellText=data, colLabels=col_labels,
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.0, 1.8)

    HS_CLR  = "#FFCDD2"   # light red header
    CS_CLR  = "#BBDEFB"   # light blue header

    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#BBBBBB"); cell.set_linewidth(0.5)
        if r == 0:
            # Column header colour coding
            if c in (2, 3):
                cell.set_facecolor(HS_CLR)
            elif c in (5, 6):
                cell.set_facecolor(CS_CLR)
            elif c == 4:
                cell.set_facecolor("#EEEEEE")
            else:
                cell.set_facecolor("#1F3864")
                cell.set_text_props(color="white")
            cell.set_text_props(fontweight="bold", fontsize=10)
        else:
            t_val = int(data[r - 1][0])
            cell.set_facecolor(TABLE_ROW_COLORS_T.get(t_val, "white"))

    ax.set_title(
        f"Gi* classification summary — {lot_label}\n"
        f"(distance threshold = {distance_threshold} m; "
        "α = 0.05 / 0.01 two-tailed)",
        fontsize=11, fontweight="bold", pad=14, color="black",
    )
    plt.tight_layout()
    _save(fig, f"Table_GiStar_{lot_label.replace(' ', '_')}")


# =============================================================================
# 7. MAIN PIPELINE
# =============================================================================

if __name__ == "__main__":

    lots_config = [
        (DONMATIAS_XLSX, "Donmatias", True),    # has_utm=True
        (EL_RETIRO_XLSX, "El Retiro", False),   # has_utm=False (Lat/Lon)
        (LA_CEJA_XLSX,   "La Ceja",   False),
    ]

    for xlsx, label, has_utm in lots_config:

        if xlsx is None or not os.path.exists(xlsx):
            print(f"\n[{label}] file not found — skipped.")
            continue

        print(f"\n{'='*65}")
        print(f"  LOT: {label}")
        print(f"{'='*65}")

        # ── Step 1: load data ──────────────────────────────────────────────────
        df = load_lot(xlsx, label, has_utm=has_utm)

        # ── Step 2: set distance threshold ────────────────────────────────────
        # Use per-lot value from config dict; fall back to global if defined
        try:
            d_thresh = GLOBAL_THRESHOLD
            print(f"  Distance threshold (global): {d_thresh} m")
        except NameError:
            d_thresh = DISTANCE_THRESHOLDS.get(label, 45.0)
            print(f"  Distance threshold (variogram range): {d_thresh} m")

        # ── Step 3: compute Gi* for all display time points ───────────────────
        time_pts_avail = [t for t in TIME_POINTS_SHOW
                          if f"D{t}" in df.columns]
        print(f"\n  Computing Gi* for {len(time_pts_avail)} time points:")
        gi_results, summary = run_gi_star_all_periods(
            df, time_pts_avail, d_thresh, label)

        # ── Step 4: export full results CSV ───────────────────────────────────
        export_results_csv(df, gi_results, label)

        # Save summary CSV
        summary_path = os.path.join(
            OUTPUT_DIR,
            f"gistar_summary_{label.replace(' ', '_')}.csv")
        pd.DataFrame(summary).to_csv(summary_path, index=False)
        print(f"  Summary saved: {summary_path}")

        # ── Step 5: figures ────────────────────────────────────────────────────
        print("\n  Generating figures:")
        figure_gistar_maps(df, gi_results, label, d_thresh)
        figure_summary_table(summary, label, d_thresh)

    print("\nAll analyses complete.")